In [7]:
import gymnasium as gym
import torch
import torch.optim as opti
import torch.nn as nn
from torch.distributions import Normal
import numpy as np

### Creating an Gym Enivronment with Continous Action Space

In [4]:
env = gym.make('Pendulum-v1')

state_dim = env.observation_space.shape[0]
action_dim = env.action_space.shape[0]
action_low = float(env.action_space.low[0])
action_high = float(env.action_space.high[0])

### Creating the architecture of Actor Network

In [19]:
from matplotlib.pyplot import axis
class Actor(nn.Module):
    def __init__(self, state_dim, action_dim, action_high, hidden_dim = 256):
        super ().__init__()
        self.action_high = action_high
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU()
        )
        self.mean_calc = nn.Linear(128, action_dim)
        self.log_std_calc = nn.Linear(128, action_dim)
    
    def forward(self, state):
        features = self.net(state)
        mean = self.mean_calc(features)
        log_std = self.log_std_calc(features)
        log_std = torch.clamp(log_std, min = -20, max = 5)
        std = torch.exp(log_std)
        return Normal(mean , std)
    
    def get_action(self, state):
        distrib = self.forward(state)
        raw_action = distrib.rsample()
        squashed_action = torch.tanh(raw_action)
        scaled_action = squashed_action * self.action_high
        log_prob = distrib.log_prob(raw_action).sum(axis = -1, keepdim = True)
        log_prob -= torch.log(1 - squashed_action.pow(2) + 1e-6).sum(axis= -1, keepdim = True)
        return scaled_action, log_prob

### Creating the architecture of Critic Network

In [20]:
class Critic(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim = 256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim + action_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU6(),
            nn.Linear(256, 1)
        )
    def forward(self, state, action):
        x = torch.cat([state, action], dim =1)
        return self.net(x)

In [21]:
batch_size = 32
alpha = 0.2
gamma = 0.99

Creating the Target Networks as well, respectively alongwith initiating the ADAM Optimizer for both

In [40]:
actor = Actor(state_dim, action_dim, action_high)
critic = Critic(state_dim, action_dim)

# Target critic network used to calculate the stable target_q (Bellman equation)
target_critic = Critic(state_dim, action_dim)
target_critic.load_state_dict(critic.state_dict())

actor_optimizer = opti.Adam(actor.parameters(), lr=3e-4)
critic_optimizer = opti.Adam(critic.parameters(), lr=3e-4)

# 4. Generate Real Data from the Environment to form a training batch
state_list, action_list, reward_list, next_state_list = [], [], [], []
state, info = env.reset()

Writing the training loop

In [41]:
for _ in range(batch_size):
    # Convert single state to tensor and add batch dimension [1, STATE_DIM]
    state_tensor = torch.FloatTensor(state).unsqueeze(0)
    
    # Get action from actor, convert to a format Gym understands
    with torch.no_grad():
        scaled_action, _ = actor.get_action(state_tensor)
    action_for_env = scaled_action.squeeze(0).numpy()
    
    # Step environment
    next_state, reward, terminated, truncated, info = env.step(action_for_env)
    
    # Save step transitions to build our batch
    state_list.append(state)
    action_list.append(action_for_env)
    reward_list.append([reward])
    next_state_list.append(next_state)
    
    state = next_state if not (terminated or truncated) else env.reset()[0]

Training for the epochs!

In [55]:
# Convert lists of steps into proper batch tensors of shape [BATCH_SIZE, ...]
states = torch.FloatTensor(state_list)
actions = torch.FloatTensor(action_list)
rewards = torch.FloatTensor(reward_list)
next_states = torch.FloatTensor(next_state_list)

# 5. Run a Single Training Step
# --- CALCULATE BELLMAN TARGET FOR CRITIC ---
for i in range(100000):
    with torch.no_grad():
        # Predict what action the actor would take next
        next_actions, next_log_probs = actor.get_action(next_states)
        # Evaluate it with the target critic
        next_q = target_critic(next_states, next_actions)
        # Target Q = Reward + Gamma * (Next Q - Alpha * Entropy)
        target_q = rewards + gamma * (next_q - alpha * next_log_probs)

### Updates and the loss Calculation:

In [54]:
# --- CRITIC UPDATE ---
predicted_q = critic(states, actions)
critic_loss = nn.MSELoss()(predicted_q, target_q)

critic_optimizer.zero_grad()
critic_loss.backward()
critic_optimizer.step()

# --- ACTOR UPDATE ---
new_actions, log_probs = actor.get_action(states)
q_value_of_new_actions = critic(states, new_actions)

# Objective: Maximize Q-value while maximizing distribution entropy
actor_loss = -(q_value_of_new_actions - alpha * log_probs).mean()

actor_optimizer.zero_grad()
actor_loss.backward()
actor_optimizer.step()

print(f"Success! Environment data gathered.")
print(f"Critic Loss: {critic_loss.item():.4f} | Actor Loss: {actor_loss.item():.4f}")

env.close()

Success! Environment data gathered.
Critic Loss: 16.3923 | Actor Loss: 1.8871
